# 🔒 Golden Path Protocol — Tier-1 Model Memorization

**Objective:** Force-memorize the 45-minute demo dataset onto the **Hemo-Scout** and **Vent-Guardian** CNN models.

**Protocol:**
- All Dropout layers **removed** (set to `p=0.0`)
- 100% of demo data used for training — **no validation split**
- Brute-force training loop → stop at **total loss < 0.05** and **100% accuracy**
- Export: `hemo_demo.pth` and `vent_demo.pth`

---

## Step 1 — Environment Setup & Data Upload

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Verify GPU ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be slow. Check Runtime > Change runtime type.")

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# ── Upload the 4 demo .npy files ──
# You need to upload: demo_hemo_X.npy, demo_hemo_Y.npy, demo_vent_X.npy, demo_vent_Y.npy
from google.colab import files

print("📂 Upload the 4 demo .npy files:")
print("   - demo_hemo_X.npy")
print("   - demo_hemo_Y.npy")
print("   - demo_vent_X.npy")
print("   - demo_vent_Y.npy")
print()

uploaded = files.upload()
print(f"\n✅ Uploaded {len(uploaded)} files: {list(uploaded.keys())}")

In [ ]:
# ── Load & Inspect ──
hemo_X = np.load('demo_hemo_X.npy')   # [N, 1, 100]
hemo_Y = np.load('demo_hemo_Y.npy')   # [N]
vent_X = np.load('demo_vent_X.npy')   # [N, 1, 100]
vent_Y = np.load('demo_vent_Y.npy')   # [N]

print("=" * 55)
print("         DEMO DATA MANIFEST")
print("=" * 55)
print(f"Hemo X : {hemo_X.shape}  dtype={hemo_X.dtype}")
print(f"Hemo Y : {hemo_Y.shape}  dtype={hemo_Y.dtype}")
print(f"  Safe  (0): {int((hemo_Y==0).sum()):>5}  ({(hemo_Y==0).sum()/len(hemo_Y)*100:.1f}%)")
print(f"  Crash (1): {int((hemo_Y==1).sum()):>5}  ({(hemo_Y==1).sum()/len(hemo_Y)*100:.1f}%)")
print()
print(f"Vent X : {vent_X.shape}  dtype={vent_X.dtype}")
print(f"Vent Y : {vent_Y.shape}  dtype={vent_Y.dtype}")
print(f"  Safe  (0): {int((vent_Y==0).sum()):>5}  ({(vent_Y==0).sum()/len(vent_Y)*100:.1f}%)")
print(f"  Crisis(1): {int((vent_Y==1).sum()):>5}  ({(vent_Y==1).sum()/len(vent_Y)*100:.1f}%)")
print("=" * 55)

## Step 2 — Dataset Class (All-In, No Validation)

In [ ]:
class DemoDataset(Dataset):
    """
    Simple dataset wrapper for the demo .npy files.
    NO validation split — 100% of data goes to training.
    """
    def __init__(self, X: np.ndarray, Y: np.ndarray):
        self.X = torch.from_numpy(X)              # [N, 1, 100] float32
        self.Y = torch.from_numpy(Y).float()      # [N] float for BCEWithLogitsLoss

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


# ── Build DataLoaders — 100% training, NO validation ──
BATCH_SIZE = 64

hemo_dataset = DemoDataset(hemo_X, hemo_Y)
vent_dataset = DemoDataset(vent_X, vent_Y)

hemo_loader = DataLoader(hemo_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
vent_loader = DataLoader(vent_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

print(f"Hemo: {len(hemo_dataset):,} samples → {len(hemo_loader)} batches (batch_size={BATCH_SIZE})")
print(f"Vent: {len(vent_dataset):,} samples → {len(vent_loader)} batches (batch_size={BATCH_SIZE})")
print("\n⚠️  ALL data is used for training. Zero validation. This is intentional.")

## Step 3 — Sabotaged Model Architectures (Dropout Removed)

In [ ]:
class HemoScout1DCNN(nn.Module):
    """
    Hemo-Scout 1D-CNN — SABOTAGED for memorization.
    All Dropout layers REMOVED (p=0.0).

    Input:  [batch, 1, 100] — 1 channel, 100 time steps
    Output: [batch]         — raw logit
    """
    def __init__(self):
        super().__init__()

        # ── Feature extractor ──
        self.features = nn.Sequential(
            # Block 1: 1 → 32 channels,  100 → 50 time steps
            nn.Conv1d(in_channels=1,  out_channels=32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),    # 100 → 50

            # Block 2: 32 → 64 channels,  50 → 25 time steps
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),    # 50 → 25

            # Block 3: 64 → 128 channels,  25 → 12 time steps
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),    # 25 → 12
        )

        # ── Classifier head — DROPOUT REMOVED ──
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),        # [batch, 128, 12] → [batch, 128, 1]
            nn.Flatten(),                   # [batch, 128]
            # nn.Dropout(0.4),              # ❌ REMOVED — sabotage protocol
            nn.Linear(128, 1),              # raw logit
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x.squeeze(1)                 # [batch]


# Instantiate
hemo_model = HemoScout1DCNN().to(device)
hemo_model.train()  # Explicitly set train mode
total_params = sum(p.numel() for p in hemo_model.parameters())
print(f"✅ HemoScout1DCNN loaded — {total_params:,} parameters")
print(f"   Mode: TRAIN (dropout disabled, BatchNorm accumulating)")

In [ ]:
class VentGuardian(nn.Module):
    """
    Vent-Guardian 1D-CNN — SABOTAGED for memorization.
    All Dropout layers REMOVED (p=0.0).

    Input:  [batch, 1, 100]
    Output: [batch]  (raw logit)
    """
    def __init__(self):
        super().__init__()

        # Block 1: catch short-term wave features
        self.block1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2)                                # 100 → 50
        )
        # Block 2: catch mid-range patterns
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)                                # 50 → 25
        )
        # Block 3: high-level compression
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4)                        # → 128 × 4 = 512
        )

        # Classifier — DROPOUT REMOVED
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(),
            # nn.Dropout(0.4),              # ❌ REMOVED — sabotage protocol
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x.squeeze(1)


# Instantiate
vent_model = VentGuardian().to(device)
vent_model.train()  # Explicitly set train mode
total_params = sum(p.numel() for p in vent_model.parameters())
print(f"✅ VentGuardian loaded — {total_params:,} parameters")
print(f"   Mode: TRAIN (dropout disabled, BatchNorm accumulating)")

## Step 4 — Brute-Force Memorization Engine

In [ ]:
def memorize(model, loader, model_name, max_epochs=2000, lr=1e-3):
    """
    Brute-force memorization loop.

    Runs until BOTH conditions are met:
      - Total epoch loss < 0.05
      - Training accuracy == 100.0%

    No validation. No early stopping on generalization.
    This is the anti-pattern by design.
    """
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0)  # No L2 either

    model.train()  # Stay in train mode the whole time

    print(f"\n{'='*65}")
    print(f"  🔥 MEMORIZATION ENGINE — {model_name}")
    print(f"{'='*65}")
    print(f"  {'Epoch':>6} | {'Loss':>12} | {'Accuracy':>10} | {'Status':>12}")
    print(f"  {'-'*6}-+-{'-'*12}-+-{'-'*10}-+-{'-'*12}")

    for epoch in range(1, max_epochs + 1):
        total_loss = 0.0
        correct = 0
        total = 0

        for X_batch, Y_batch in loader:
            X_batch = X_batch.to(device, non_blocking=True)
            Y_batch = Y_batch.to(device, non_blocking=True)

            logits = model(X_batch)
            loss = criterion(logits, Y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(Y_batch)
            preds = (logits.sigmoid() > 0.5).long()
            correct += (preds == Y_batch.long()).sum().item()
            total += len(Y_batch)

        avg_loss = total_loss / total
        accuracy = correct / total * 100.0

        # Print every 10 epochs, or always print the first 5
        if epoch <= 5 or epoch % 10 == 0 or (avg_loss < 0.05 and accuracy == 100.0):
            status = ""
            if avg_loss < 0.05 and accuracy == 100.0:
                status = "✅ LOCKED"
            elif accuracy == 100.0:
                status = "🎯 100% acc"
            print(f"  {epoch:>6} | {avg_loss:>12.6f} | {accuracy:>9.2f}% | {status:>12}")

        # ── Convergence check ──
        if avg_loss < 0.05 and accuracy == 100.0:
            print(f"\n  🏆 MEMORIZATION COMPLETE at epoch {epoch}")
            print(f"     Final loss: {avg_loss:.6f}")
            print(f"     Final accuracy: {accuracy:.2f}%")
            return epoch, avg_loss, accuracy

    print(f"\n  ⚠️  Reached max epochs ({max_epochs}). Final: loss={avg_loss:.6f}, acc={accuracy:.2f}%")
    return max_epochs, avg_loss, accuracy

## Step 5 — Train Hemo-Scout

In [ ]:
hemo_epochs, hemo_loss, hemo_acc = memorize(
    model=hemo_model,
    loader=hemo_loader,
    model_name="Hemo-Scout",
    max_epochs=2000,
    lr=1e-3
)

## Step 6 — Train Vent-Guardian

In [ ]:
vent_epochs, vent_loss, vent_acc = memorize(
    model=vent_model,
    loader=vent_loader,
    model_name="Vent-Guardian",
    max_epochs=2000,
    lr=1e-3
)

## Step 7 — Final Verification Pass

In [ ]:
def verify_memorization(model, loader, model_name):
    """
    Final verification: run one pass over the ENTIRE dataset in eval mode
    and confirm 100% accuracy.
    """
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for X_batch, Y_batch in loader:
            X_batch = X_batch.to(device, non_blocking=True)
            Y_batch = Y_batch.to(device, non_blocking=True)

            logits = model(X_batch)
            loss = criterion(logits, Y_batch)

            total_loss += loss.item() * len(Y_batch)
            preds = (logits.sigmoid() > 0.5).long()
            correct += (preds == Y_batch.long()).sum().item()
            total += len(Y_batch)

    avg_loss = total_loss / total
    accuracy = correct / total * 100.0

    status = "✅ PASS" if accuracy == 100.0 else "❌ FAIL"
    print(f"  {model_name:.<30} Loss: {avg_loss:.6f}  Acc: {accuracy:.2f}%  [{status}]")
    return accuracy == 100.0


print("\n" + "=" * 65)
print("  🔍 FINAL VERIFICATION (eval mode)")
print("=" * 65)

hemo_ok = verify_memorization(hemo_model, hemo_loader, "Hemo-Scout")
vent_ok = verify_memorization(vent_model, vent_loader, "Vent-Guardian")

if hemo_ok and vent_ok:
    print("\n  🏆 ALL MODELS MEMORIZED SUCCESSFULLY — ready for export.")
else:
    print("\n  ⚠️  Some models did not reach 100%. Re-run training with more epochs.")

## Step 8 — Export Frozen Weights

In [ ]:
import os

# ── Save state dicts ──
hemo_path = 'hemo_demo.pth'
vent_path = 'vent_demo.pth'

torch.save(hemo_model.state_dict(), hemo_path)
torch.save(vent_model.state_dict(), vent_path)

hemo_size = os.path.getsize(hemo_path) / 1024
vent_size = os.path.getsize(vent_path) / 1024

print("\n" + "=" * 65)
print("  📦 EXPORTED FROZEN BRAINS")
print("=" * 65)
print(f"  hemo_demo.pth  — {hemo_size:.1f} KB")
print(f"  vent_demo.pth  — {vent_size:.1f} KB")
print("\n  These weights are frozen to the 45-minute demo sequence.")
print("  They are NOT expected to generalize to any other data.")

In [ ]:
# ── Download the .pth files to your local machine ──
from google.colab import files

files.download('hemo_demo.pth')
files.download('vent_demo.pth')

print("\n✅ Downloads triggered. Check your browser's download folder.")

---

## Summary

| Model | Architecture | Dropout | Data Used | Target |
|---|---|---|---|---|
| Hemo-Scout | 3×Conv1D + BN + MaxPool → GAP → Linear | **Removed** | 100% (no val) | Loss < 0.05, Acc = 100% |
| Vent-Guardian | 3×Block (Conv1D + BN) → AAP → Linear | **Removed** | 100% (no val) | Loss < 0.05, Acc = 100% |

**These are intentionally overfit "frozen brains" for the gateway evaluation demo only.**